# Single-node IE: IG vs exact patching

**EXPLORATORY** (CLAUDE.md §2: exempt from the TDD gate).

Compares, node by node, two estimates of the same quantity, IE(n) = m(clean | do(n = n_patch)) − m(clean):

- **exact**: `scripts/exact_ie.py --size 1` (real clamp patching, every *moving* node).
- **IG**: the `node_attribution` stage, grad·(f_patch − f_clean) with integrated gradients.

Nodes are matched on (example prompt, site, token, feature); error nodes are `feature_idx = -1`.
`diff = IG − exact`.

**The statistical test.** For each token position, H0: the mean of IG − exact is zero. The unit is the
**example**: nodes from one prompt share a forward pass and are not independent, so the main test is a
one-sample t-test on the per-example mean of IG − exact (n = number of examples). The node-level t-test
treats every node as independent. It is shown for contrast, and with ~10⁵ nodes per position it calls
almost any difference significant. p-values are adjusted across token positions with `ADJUST`.

In [ ]:
import json
import os
import sys
from pathlib import Path
from types import SimpleNamespace

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display
from scipy import stats
from statsmodels.stats.multitest import multipletests
from transformers import AutoTokenizer

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT)                      # config data paths are repo-relative
sys.path.insert(0, str(ROOT))
from src.attribution.nodes import read_node_hive
from src.data.loaders import load_examples

EXACT = Path("outputs/exact_ie/size1")
IG = Path("outputs/node_attribution/b21820f5")   # rc_train, 100 examples, ig_steps=10
THRESH = 0.0         # drop a node if |IG| < THRESH and |exact| < THRESH (0 keeps all)
ALPHA = 0.05
ADJUST = "holm"      # statsmodels multipletests method: "holm" (FWER), "fdr_bh" (FDR), ...
SYMLOG = 1e-4        # linthresh for symlog y-axes on the box plots; None for linear
KEY = ["site_ordinal", "token_idx", "feature_idx"]
TOK = AutoTokenizer.from_pretrained("EleutherAI/pythia-70m-deduped")

## Match examples across the two runs

In [ ]:
def replay(data_cfg, seed):
    # Re-run the loader as the run did. Example indices are only meaningful under
    # their own run's config (num_examples + use_min_length_only change which
    # examples survive), so the runs are matched by prompt text, not index.
    # The loader only uses .tokenizer.
    return load_examples(SimpleNamespace(**data_cfg), SimpleNamespace(tokenizer=TOK), seed)

meta_x = json.loads((EXACT / "metadata.json").read_text(encoding="utf-8"))
meta_ig = json.loads((IG / "manifest.json").read_text(encoding="utf-8"))["params"]
cfg_x = meta_x["config"]
assert cfg_x["suite"]["name"] == meta_ig["suite"]["name"], (cfg_x["suite"], meta_ig["suite"])
assert cfg_x["attribution"]["metric"] == meta_ig["attribution"]["metric"]
assert meta_ig["attribution"]["method"] == "ig", meta_ig["attribution"]

ex_x = replay(cfg_x["data"], cfg_x["seed"])
# The IG manifest predates some data fields; fill them from the exact run's config.
ex_ig = replay({**cfg_x["data"], **meta_ig["data"]}, meta_ig["seed"])
ig_idx = {(e.clean_prefix, e.patch_prefix): k for k, e in enumerate(ex_ig)}

done = sorted(int(p.parent.name[2:]) for p in EXACT.glob("ex*/singles.parquet"))
matched = [(k, ig_idx[(ex_x[k].clean_prefix, ex_x[k].patch_prefix)]) for k in done
           if (ex_x[k].clean_prefix, ex_x[k].patch_prefix) in ig_idx]
print(f"exact: {len(done)} examples done; IG: {len(ex_ig)} examples "
      f"(ig_steps={meta_ig['attribution']['ig_steps']}); matched: {len(matched)}")

## Load and join node IEs

In [ ]:
ig_all = read_node_hive(IG)
ig_by_ex = {k: g for k, g in ig_all.groupby("example_idx")}

parts, n_missing = [], 0
for kx, kig in matched:
    d = EXACT / f"ex{kx:05d}"
    n = pd.read_parquet(d / "nodes.parquet")
    s = pd.read_parquet(d / "singles.parquet").sort_values("node")
    assert (s.node.to_numpy() == np.arange(len(n))).all()
    n["exact"] = s.ie.to_numpy()
    ig = ig_by_ex.get(kig, ig_all.iloc[:0])[KEY + ["attribution"]].rename(columns={"attribution": "ig"})
    n = n.merge(ig, on=KEY, how="left", validate="one_to_one")
    n_missing += int(n.ig.isna().sum())
    parts.append(n.dropna(subset=["ig"]).assign(example=kx))

df = pd.concat(parts, ignore_index=True)
df["node_type"] = np.where(df.feature_idx < 0, "error", "feature")
df = df[(df.ig.abs() >= THRESH) | (df.exact.abs() >= THRESH)].assign(
    diff=lambda d: d.ig.astype(np.float64) - d.exact)
print(f"{len(df):,} nodes kept; {n_missing:,} moving nodes had no IG row and were dropped")

# Tick labels: the first matched example's tokens. Positions only line up across
# prompts when all have one length (use_min_length_only=True).
ids = ig_by_ex[matched[0][1]].drop_duplicates("token_idx").set_index("token_idx").token_id
POS = sorted(df.token_idx.unique())
POS_LABEL = [f"{t}\n{TOK.convert_ids_to_tokens(int(ids[t]))!r}" if t in ids else str(t) for t in POS]

## Key statistics

`R2_y=x` treats exact as ground truth: 1 − Σ(IG − exact)² / Σ(exact − mean exact)².
`rel_abs_err` = Σ|IG − exact| / Σ|exact|. `sign_agree` counts exact zeros as a sign of their own.

In [ ]:
def summary(g):
    x, y, d = (g[c].to_numpy(np.float64) for c in ("exact", "ig", "diff"))
    return pd.Series({
        "n_nodes": len(g), "n_examples": g.example.nunique(),
        "mean_exact": x.mean(), "mean_ig": y.mean(), "mean_diff": d.mean(),
        "median_exact": np.median(x), "median_ig": np.median(y), "median_diff": np.median(d),
        "std_exact": x.std(ddof=1), "std_ig": y.std(ddof=1), "std_diff": d.std(ddof=1),
        "MAE": np.abs(d).mean(), "rel_abs_err": np.abs(d).sum() / np.abs(x).sum(),
        "pearson_r": stats.pearsonr(x, y).statistic,
        "spearman_rho": stats.spearmanr(x, y).statistic,
        "R2_y=x": 1 - (d ** 2).sum() / ((x - x.mean()) ** 2).sum(),
        "sign_agree": (np.sign(x) == np.sign(y)).mean(),
    })

def show(t):
    ints = ["n_nodes", "n_examples"]
    display(t.style.format("{:.4g}").format("{:,.0f}", subset=[c for c in ints if c in t]))

overall = pd.DataFrame({"all": summary(df),
                        **{k: summary(g) for k, g in df.groupby("node_type")}}).T
display(Markdown("**Pooled over all nodes, and by node type**"))
show(overall)
per_pos = df.groupby("token_idx").apply(summary, include_groups=False)
display(Markdown("**Per token position**"))
show(per_pos)

## Box plots

In [ ]:
BOX = dict(showmeans=True, meanprops=dict(marker="^", mfc="white", mec="k", ms=5),
           flierprops=dict(marker=".", ms=1.5, alpha=0.3, rasterized=True))
COL = {"exact": "C0", "ig": "C1", "diff": "C2"}

def style(ax, ylabel):
    if SYMLOG:
        ax.set_yscale("symlog", linthresh=SYMLOG)
    ax.axhline(0, color="k", lw=0.6)
    ax.set_ylabel(ylabel)

def boxes(ax, data, positions, color, width):
    b = ax.boxplot(data, positions=positions, widths=width, patch_artist=True, **BOX)
    for p in b["boxes"]:
        p.set(facecolor=color, alpha=0.6)

fig, (a1, a2) = plt.subplots(1, 2, figsize=(9, 5), width_ratios=[2, 1])
for i, c in enumerate(("exact", "ig")):
    boxes(a1, [df[c]], [i], COL[c], 0.6)
a1.set_xticks([0, 1], ["exact", "IG"])
style(a1, "IE")
boxes(a2, [df["diff"]], [0], COL["diff"], 0.6)
a2.set_xticks([0], ["IG − exact"])
style(a2, "IG − exact")
fig.suptitle(f"All nodes, all positions (n = {len(df):,}; ▲ = mean)")
fig.tight_layout()
plt.show()

x = np.arange(len(POS))
fig, (a1, a2) = plt.subplots(2, 1, figsize=(max(8, 1.1 * len(POS)), 9), sharex=True)
for off, c in ((-0.2, "exact"), (0.2, "ig")):
    boxes(a1, [df.loc[df.token_idx == t, c] for t in POS], x + off, COL[c], 0.35)
a1.legend([plt.Rectangle((0, 0), 1, 1, fc=COL[c], alpha=0.6) for c in ("exact", "ig")],
          ["exact", "IG"], loc="upper right")
style(a1, "IE")
boxes(a2, [df.loc[df.token_idx == t, "diff"] for t in POS], x, COL["diff"], 0.5)
style(a2, "IG − exact")
a2.set_xticks(x, POS_LABEL)
a2.set_xlabel("token position (tokens of the first matched example)")
fig.suptitle("Per token position (▲ = mean)")
fig.tight_layout()
plt.show()

## Is the mean IG − exact zero? (per token position, multiplicity-adjusted)

`mean_diff`, `ci_low/high`, `t`, `p` are from the example-level test (per-example mean of IG − exact,
n = examples). `p_adj` adjusts those across positions with `ADJUST`; `significant` = `p_adj < ALPHA`.
`p_node_adj` is the node-level test, adjusted the same way, for contrast only.

In [ ]:
def ttest(g):
    per_ex = g.groupby("example")["diff"].mean()
    r = stats.ttest_1samp(per_ex, 0.0)
    ci = r.confidence_interval(1 - ALPHA)
    return pd.Series({"n_examples": len(per_ex), "n_nodes": len(g),
                      "mean_diff": per_ex.mean(), "ci_low": ci.low, "ci_high": ci.high,
                      "t": r.statistic, "p": r.pvalue,
                      "p_node": stats.ttest_1samp(g["diff"], 0.0).pvalue})

tests = df.groupby("token_idx").apply(ttest, include_groups=False)
tests = tests[tests.n_examples >= 2]      # a t-test needs two examples
tests["p_adj"] = multipletests(tests.p, alpha=ALPHA, method=ADJUST)[1]
tests["p_node_adj"] = multipletests(tests.p_node, alpha=ALPHA, method=ADJUST)[1]
tests["significant"] = tests.p_adj < ALPHA
tests.insert(0, "token", [POS_LABEL[POS.index(t)].split("\n")[-1] for t in tests.index])

display(tests.style.format("{:.4g}", subset=["mean_diff", "ci_low", "ci_high", "t", "p", "p_adj",
                                             "p_node", "p_node_adj"])
             .format("{:,.0f}", subset=["n_examples", "n_nodes"]))

pooled = ttest(df)
display(Markdown(
    f"**All positions pooled** (single test, not part of the family): mean IG − exact = "
    f"{pooled.mean_diff:.4g}, {1 - ALPHA:.0%} CI [{pooled.ci_low:.4g}, {pooled.ci_high:.4g}], "
    f"t = {pooled.t:.3g}, p = {pooled.p:.3g} (n = {pooled.n_examples:.0f} examples)."))